# 03 Lines and regression: NIST certified fits, Anscombe, mtcars

simplydata's words for fitting lines and models by least squares, checked on
- the eleven **NIST StRD linear regression** datasets, certified to 15 digits: Norris (a straight line), Pontius (a
  quadratic), Wampler1 to Wampler5 (degree 5), Filip (degree 10, notoriously ill-conditioned), Longley (six
  predictors, highly correlated), NoInt1 and NoInt2 (a line through 0);
- **Anscombe's quartet** (1973): four datasets with the same printed fit;
- **mtcars** (Henderson & Velleman 1981) and **stackloss** (Brownlee 1965).

**Only simplydata runs in this notebook.** The reference values are in `references/03_regression_NIST.csv`, written
by `scripts/references/03_regression_NIST.py`: the NIST certified values (at least 9 significant digits must agree,
LRE), Anscombe's printed fit, and, only for mtcars and stackloss, where nothing is printed, R `lm()` run once there
(relative difference 1e-9). Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows_by, versions

NOTEBOOK = "03_regression_NIST"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## A straight line: `table.fit_line` on NIST Norris

The same underneath: SciPy `stats.linregress`.

In [2]:
result = load("nist_Norris").fit_line(x="x", y="y")

"y" goes up by 1.00 for each 1 more "x" (95 % CI 1.00 to 1.00). R squared = 1.00: "x" explains about 100 % of the differences in "y".
If there were truly no slope, a slope this steep or steeper would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Line: y = 1.00 * x + -0.262, by least squares over 36 rows. A link does not show that one thing causes the other.


In [3]:
row = read_row(result)
checks.check_values("NIST Norris", "fit_line", {q: row[q] for q in ["intercept", "slope", "intercept standard error",
                                                                     "slope standard error", "R squared"]})

agrees  NIST Norris | fit_line | intercept: simplydata -0.262323073774, NIST certified -0.262323073774; LRE 12.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Norris.shtml
agrees  NIST Norris | fit_line | slope: simplydata 1.00211681802, NIST certified 1.00211681802; LRE 14.4 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Norris.shtml
agrees  NIST Norris | fit_line | intercept standard error: simplydata 0.232818234302, NIST certified 0.232818234301; LRE 11.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Norris.shtml
agrees  NIST Norris | fit_line | slope standard error: simplydata 0.000429796848201, NIST certified 0.0004297968482; LRE 11.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Norris.shtml
agrees  NIST Norris | fit_line | R squared: simplydata 0.999993745884,

5

## Polynomials: `table.fit_curve(of_degree=)` on Pontius, Wampler1-5 and Filip

Least squares on the powers of x. simplydata (after 0.2.0) maps x onto [-1, 1] before taking the powers, fits
statsmodels WLS there and maps the estimates and their covariance back to the raw powers (the fix of FINDINGS A1).
Each estimate, its standard error and R squared are compared with the certified values.

In [4]:
POLYNOMIALS = {"Pontius": 2, "Wampler1": 5, "Wampler2": 5, "Wampler3": 5, "Wampler4": 5, "Wampler5": 5, "Filip": 10}
load("nist_Pontius").fit_curve(x="x", y="y", of_degree=2)

Curve: y = 0.000674 + 7.32e-07 * x - 3.16e-15 * x**2 (each with its 95 % CI in the table); R squared = 1.00, adjusted 1.00.
Method: least squares on powers of x, as R lm(y ~ poly(x, 2, raw = TRUE)).


,term,estimate,standard error,statistic name,statistic,degrees of freedom,p-value,95 % CI low,95 % CI high,R squared,adjusted R squared,rows
1,(intercept),6.735658e-04,1.079386e-04,t,6.240267,37.0,2.970542e-07,4.548614e-04,8.922702e-04,1.0,1.0,40
2,x,7.320592e-07,1.578174e-10,t,4638.646692,37.0,2.952199e-108,7.317394e-07,7.323789e-07,1.0,1.0,40
3,x**2,-3.160819e-15,4.866528e-17,t,-64.950174,37.0,9.835634e-40,-3.259424e-15,-3.062213e-15,1.0,1.0,40


In [5]:
for name, degree in POLYNOMIALS.items():
    found = read_rows_by(quiet(load(f"nist_{name}").fit_curve, x="x", y="y", of_degree=degree), "term")
    terms = ["(intercept)", "x"] + [f"x**{k}" for k in range(2, degree + 1)]
    values = {}
    for term in terms:
        values[f"{term} estimate"] = found[term]["estimate"]
        values[f"{term} standard error"] = found[term]["standard error"]
    values["R squared"] = found["(intercept)"]["R squared"]
    checks.check_values(f"NIST {name}", "fit_curve", values)

agrees  NIST Pontius | fit_curve | (intercept) estimate: simplydata 0.000673565789473, NIST certified 0.000673565789474; LRE 12.2 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve | (intercept) standard error: simplydata 0.000107938612033, NIST certified 0.000107938612033; LRE 12.1 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve | x estimate: simplydata 7.32059160401e-07, NIST certified 7.32059160401e-07; LRE 14.9 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve | x standard error: simplydata 1.57817399982e-10, NIST certified 1.57817399982e-10; LRE 13.1 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | f

## Several predictors: `table.explain` on NIST Longley

Employment from six economic series, 1947-1962 (Longley 1967), a classic test of accuracy because the predictors are
almost collinear. `explain` with numbers fits R's `lm()`; the same underneath: statsmodels `OLS(...).fit(method="qr")`.

In [6]:
USING = [f"x{i}" for i in range(1, 7)]
result = load("nist_Longley").explain("y", using=USING)

Explaining "y" from "x1", "x2", "x3", "x4", "x5" and "x6" (16 rows). Each effect below holds the other columns fixed:
  intercept: -3.48e+06, the expected "y" when every number column is 0.
  x1: each 1 more "x1" goes with 15.1 more "y" (95 % CI -177 to 207); about 86 of 100 studies would show this much if there were truly none (p = 0.863).
  x2: each 1 more "x2" goes with 0.0358 less "y" (95 % CI -0.112 to 0.0399); about 31 of 100 studies would show this much if there were truly none (p = 0.313).
  x3: each 1 more "x3" goes with 2.02 less "y" (95 % CI -3.13 to -0.915); about 3 of 1000 studies would show this much if there were truly none (p = 0.003).
  x4: each 1 more "x4" goes with 1.03 less "y" (95 % CI -1.52 to -0.549); fewer than 1 in 1000 studies would show this much if there were truly none (p = 0.0009).
  x5: each 1 more "x5" goes with 0.0511 less "y" (95 % CI -0.563 to 0.460); about 83 of 100 studies would show this much if there were truly none (p = 0.826).
  x6: each 1 more 

In [7]:
found = read_rows_by(result, "term")
values = {}
for term in ["(intercept)"] + USING:
    values[f"{term} effect"] = found[term]["effect"]
    values[f"{term} standard error"] = found[term]["standard error"]
checks.check_values("NIST Longley", "explain", values)

agrees  NIST Longley | explain | (intercept) effect: simplydata -3482258.6346, NIST certified -3482258.6346; LRE 12.2 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Longley.shtml
agrees  NIST Longley | explain | (intercept) standard error: simplydata 890420.383608, NIST certified 890420.383607; LRE 12.6 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Longley.shtml
agrees  NIST Longley | explain | x1 effect: simplydata 15.0618722716, NIST certified 15.0618722714; LRE 10.9 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Longley.shtml
agrees  NIST Longley | explain | x1 standard error: simplydata 84.9149257748, NIST certified 84.9149257748; LRE 12.4 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Longley.shtml
agrees  NIST Longley | explain | x2 effect: simplydata -0.03581917

14

## A line through 0: NoInt1 and NoInt2

simplydata has no option for a straight line forced through 0, so these two are fitted with
`table.fit_formula("y = b1 * x")`, its general least-squares word (iterative, as R `nls()`). The model is linear, so it
must still reach the certified slope, its standard error and the residual standard deviation.

In [8]:
for name in ("NoInt1", "NoInt2"):
    row = read_row(quiet(load(f"nist_{name}").fit_formula, "y = b1 * x"))
    checks.check_values(f"NIST {name}", "fit_formula (y = b1 x)",
                        {q: row[q] for q in ["b1", "standard error of b1", "residual standard error"]})

agrees  NIST NoInt1 | fit_formula (y = b1 x) | b1: simplydata 2.07438016529, NIST certified 2.07438016529; LRE 14.8 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/NoInt1.shtml
explained difference – precision limit (B4)  NIST NoInt1 | fit_formula (y = b1 x) | standard error of b1: simplydata 0.0165289256528, NIST certified 0.0165289256198; LRE 8.7 (needs >= 9)
      why: Polynomials and a line through 0 at 7.9 to 8.9 digits - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b4-polynomials-and-a-line-through-0-at-79-to-89-digits
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/NoInt1.shtml
agrees  NIST NoInt1 | fit_formula (y = b1 x) | residual standard error: simplydata 3.56753034006, NIST certified 3.56753034006; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/lls/data/NoInt1.shtml
agrees  NIST NoInt2 | fit_formula 

## Anscombe's quartet: `table.fit_line`

Anscombe (1973) printed one fit for all four sets: y = 3 + 0.5 x, standard error of the slope 0.118, R squared 0.667.
The sets agree with it to within about one unit of the last printed digit, so the check allows that (0.001 for the
slope, its standard error and R squared; 0.01 for the intercept).

In [9]:
anscombe = quiet(sdata.table, "../data/anscombe.csv")
for i in (1, 2, 3, 4):
    row = read_row(quiet(anscombe.fit_line, x=f"x{i}", y=f"y{i}"))
    checks.check_values(f"Anscombe set {i}", "fit_line",
                        {q: row[q] for q in ["slope", "intercept", "slope standard error", "R squared"]})

agrees  Anscombe set 1 | fit_line | slope: simplydata 0.500090909091, Anscombe (1973), printed 0.5; difference 9.09e-05 (allowed 1e-03)
      Source: Anscombe (1973), printed (printed) - https://doi.org/10.1080/00031305.1973.10478966
agrees  Anscombe set 1 | fit_line | intercept: simplydata 3.00009090909, Anscombe (1973), printed 3; difference 9.09e-05 (allowed 1e-02)
      Source: Anscombe (1973), printed (printed) - https://doi.org/10.1080/00031305.1973.10478966
agrees  Anscombe set 1 | fit_line | slope standard error: simplydata 0.117905500596, Anscombe (1973), printed 0.118; difference 9.45e-05 (allowed 1e-03)
      Source: Anscombe (1973), printed (printed) - https://doi.org/10.1080/00031305.1973.10478966
agrees  Anscombe set 1 | fit_line | R squared: simplydata 0.666542459509, Anscombe (1973), printed 0.667; difference 4.58e-04 (allowed 1e-03)
      Source: Anscombe (1973), printed (printed) - https://doi.org/10.1080/00031305.1973.10478966
agrees  Anscombe set 2 | fit_line | slop

## A model with several columns: `table.explain` on mtcars and stackloss

Fuel use (miles per gallon) from weight, horsepower and gearbox (am: 1 manual); stack loss from air flow, water
temperature and acid concentration. The same underneath: statsmodels `OLS(...).fit(method="qr")`; compared with R
`lm()`: each effect, its standard error, t statistic and p-value.

In [10]:
models = {"mtcars": ("../data/mtcars.csv", "mpg", ["wt", "hp", "am"]),
          "stackloss": ("../data/stackloss.csv", "stack_loss", ["air_flow", "water_temp", "acid_conc"])}
quiet(sdata.table, models["mtcars"][0]).explain("mpg", using=["wt", "hp", "am"])

Explaining "mpg" from "wt", "hp" and "am" (32 rows). Each effect below holds the other columns fixed:
  intercept: 34.0, the expected "mpg" when every number column is 0.
  wt: each 1 more "wt" goes with 2.88 less "mpg" (95 % CI -4.73 to -1.02); about 4 of 1000 studies would show this much if there were truly none (p = 0.004).
  hp: each 1 more "hp" goes with 0.0375 less "mpg" (95 % CI -0.0572 to -0.0178); fewer than 1 in 1000 studies would show this much if there were truly none (p = 0.0005).
  am: each 1 more "am" goes with 2.08 more "mpg" (95 % CI -0.736 to 4.90); about 14 of 100 studies would show this much if there were truly none (p = 0.141).
Method: linear model by least squares, as R lm(mpg ~ wt + hp + am).
R squared = 0.840 (adjusted 0.823): the model explains about 84 % of the differences in "mpg". The model as a whole (F = 49.0, df = 3 and 28): fewer than 1 in 1000 studies would show this much if none of the columns truly mattered (p < 0.0001).
How well it fits: AIC 156, BIC

,term,effect,95 % CI low,95 % CI high,standard error,statistic name,statistic,degrees of freedom,p-value
1,(intercept),34.002875,28.589633,39.416117,2.642659,t,12.866916,28.0,2.824030e-13
2,wt,-2.878575,-4.732324,-1.024827,0.904971,t,-3.180850,28.0,3.574031e-03
3,hp,-0.037479,-0.057155,-0.017803,0.009605,t,-3.901830,28.0,5.464023e-04
4,am,2.083710,-0.735759,4.903179,1.376420,t,1.513862,28.0,1.412682e-01


In [11]:
for dataset, (path, outcome, using) in models.items():
    found = read_rows_by(quiet(quiet(sdata.table, path).explain, outcome, using=using), "term")
    values = {}
    for term in ["(intercept)"] + using:
        for column in ("effect", "standard error", "statistic", "p-value"):
            values[f"{term} {column}"] = found[term][column]
    checks.check_values(dataset, "explain", values)

agrees  mtcars | explain | (intercept) effect: simplydata 34.0028751229, R lm 34.0028751229; relative difference 4.18e-16 (allowed 1e-09)
      Source: R lm (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars | explain | (intercept) standard error: simplydata 2.64265933699, R lm 2.64265933699; relative difference 1.18e-15 (allowed 1e-09)
      Source: R lm (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars | explain | (intercept) statistic: simplydata 12.8669157795, R lm 12.8669157795; relative difference 1.52e-15 (allowed 1e-09)
      Source: R lm (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars | explain | (intercept) p-value: simplydata 2.82403019658e-13, R lm 2.82403019658e-13; relative difference 4.00e-14 (allowed 1e-09)
      Source: R lm (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars | explain | wt effect: simplydata

## Result

In [12]:
checks.finish()

03_regression_NIST: 168 comparisons - 160 agrees, 8 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,03_regression_NIST,NIST Norris,fit_line,intercept,-0.262323,-0.262323,NIST certified,4.446443e-14,9.000000e+00,lre,1.277082e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/lls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,03_regression_NIST,NIST Norris,fit_line,slope,1.002117,1.002117,NIST certified,4.218847e-15,9.000000e+00,lre,1.437572e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/lls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,03_regression_NIST,NIST Norris,fit_line,intercept standard error,0.232818,0.232818,NIST certified,3.856915e-13,9.000000e+00,lre,1.178078e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/lls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,03_regression_NIST,NIST Norris,fit_line,slope standard error,0.000430,0.000430,NIST certified,7.480453e-16,9.000000e+00,lre,1.175934e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/lls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,03_regression_NIST,NIST Norris,fit_line,R squared,0.999994,0.999994,NIST certified,-3.330669e-16,9.000000e+00,lre,1.500000e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/lls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,03_regression_NIST,stackloss,explain,water_temp p-value,0.002630,0.002630,R lm,7.329207e-17,1.000000e-09,relative,2.786713e-14,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
164,03_regression_NIST,stackloss,explain,acid_conc effect,-0.152123,-0.152123,R lm,-1.054712e-15,1.000000e-09,relative,6.933305e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
165,03_regression_NIST,stackloss,explain,acid_conc standard error,0.156294,0.156294,R lm,-1.942890e-16,1.000000e-09,relative,1.243099e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
166,03_regression_NIST,stackloss,explain,acid_conc statistic,-0.973310,-0.973310,R lm,-7.993606e-15,1.000000e-09,relative,8.212808e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
